In [14]:
pip install ursina torch numpy

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
from google.colab import output
from IPython.display import HTML, display, JSON


USE_REAL_HEMIBRAIN = True
EDGES_CSV = "/content/traced-total-connections.csv"
NODES_CSV = "/content/traced-neurons.csv"


REAL_PAIN_NEURONS = [5813020826, 5813020827, 5813020828]
REAL_GUT_NEURONS = [5813063640, 5813063641]
REAL_MOTOR_NEURONS = [511236214, 511236215]


class HemibrainConnectome(nn.Module):
    def __init__(self, use_real_data, edges_path, nodes_path):
        super().__init__()

        self.decay_rate = 0.85
        self.threshold = 1.0

        if use_real_data and os.path.exists(edges_path) and os.path.exists(nodes_path):
            print("Loading REAL Google Hemibrain Connectome...")

            edges = pd.read_csv(edges_path)
            nodes = pd.read_csv(nodes_path)

            unique_neurons = pd.concat([edges['bodyId_pre'], edges['bodyId_post']]).unique()
            self.num_neurons = len(unique_neurons)
            print(f"Mapped {self.num_neurons} biological neurons.")

            self.id_to_idx = {body_id: idx for idx, body_id in enumerate(unique_neurons)}

            nt_map = {}
            if 'nt_type' in nodes.columns:
                nt_map = nodes.set_index('bodyId')['nt_type'].to_dict()
            else:
                print("Notice: 'nt_type' column missing from CSV.")
                print("Applying biological approximation (30% inhibitory) to stabilize the brain network...")
                for nid in unique_neurons:
                    if np.random.rand() < 0.30:
                        nt_map[nid] = 'gaba'

            pre_indices = []
            post_indices = []
            weights = []

            pre_ids = edges['bodyId_pre'].values
            post_ids = edges['bodyId_post'].values
            raw_weights = edges['weight'].values

            for i in range(len(edges)):
                pre_id = pre_ids[i]

                pre_indices.append(self.id_to_idx[pre_id])
                post_indices.append(self.id_to_idx[post_ids[i]])

                w = raw_weights[i] * 0.005

                nt = str(nt_map.get(pre_id, '')).lower()
                if 'gaba' in nt or 'glut' in nt:
                    w *= -1.0

                weights.append(w)

            indices = torch.tensor([pre_indices, post_indices], dtype=torch.long)
            values = torch.tensor(weights, dtype=torch.float32)

            print("Locating highly connected hub neurons for sensory/motor pathways...")

            top_hubs = edges['bodyId_post'].value_counts().head(10).index.tolist()


            self.pain_idx = [self.id_to_idx[top_hubs[0]], self.id_to_idx[top_hubs[1]]]
            self.gut_idx = [self.id_to_idx[top_hubs[2]], self.id_to_idx[top_hubs[3]]]
            self.motor_idx = [self.id_to_idx[top_hubs[4]], self.id_to_idx[top_hubs[5]]]

        else:
            print("Warning: CSV files not found. Generating 130k balanced synthetic proxy...")
            self.num_neurons = 130000
            num_connections = 5000000

            indices = torch.randint(0, self.num_neurons, (2, num_connections))
            values = (torch.rand(num_connections) - 0.3) * 0.1

            self.id_to_idx = {}
            for i, bio_id in enumerate(REAL_PAIN_NEURONS + REAL_GUT_NEURONS + REAL_MOTOR_NEURONS):
                self.id_to_idx[bio_id] = i

            self.pain_idx = [self.id_to_idx.get(nid, np.random.randint(0, 100)) for nid in REAL_PAIN_NEURONS]
            self.gut_idx = [self.id_to_idx.get(nid, np.random.randint(100, 200)) for nid in REAL_GUT_NEURONS]
            self.motor_idx = [self.id_to_idx.get(nid, np.random.randint(200, 300)) for nid in REAL_MOTOR_NEURONS]

        self.weights = torch.sparse_coo_tensor(indices, values, (self.num_neurons, self.num_neurons)).coalesce()
        self.membrane_potentials = torch.zeros(self.num_neurons)
        self.spikes = torch.zeros(self.num_neurons)

    def forward(self, external_stimulus):
        self.membrane_potentials *= self.decay_rate
        spikes_2d = self.spikes.unsqueeze(1)
        internal_input = torch.sparse.mm(self.weights, spikes_2d).squeeze()
        self.membrane_potentials += internal_input + external_stimulus
        self.spikes = (self.membrane_potentials >= self.threshold).float()
        self.membrane_potentials[self.spikes > 0] = 0.0
        return self.spikes

brain = HemibrainConnectome(USE_REAL_HEMIBRAIN, EDGES_CSV, NODES_CSV)

fly_pos = np.array([0.0, 1.0, 0.0])
stop_pain_btn_pos = np.array([25.0, 1.0, 25.0])
stop_diarrhea_btn_pos = np.array([-25.0, 1.0, -25.0])
fly_rotation = 0.0
fly_speed = 1.5
health = 100.0


def compute_tick(is_in_pain, is_in_diarrhea):
    global fly_pos, fly_rotation, health

    stimulus = torch.zeros(brain.num_neurons)

    if is_in_pain:
        stimulus[brain.pain_idx] = 1.5
        health -= 1.5
    elif health < 100.0:
        health = min(100.0, health + 0.2)

    if is_in_diarrhea:
        stimulus[brain.gut_idx] = 1.5

    if not is_in_pain and not is_in_diarrhea:
        stimulus += (torch.rand(brain.num_neurons) * 0.02)

    spikes = brain(stimulus)
    motor_activity = spikes[brain.motor_idx].sum().item()

    pain_stopped = False
    diarrhea_stopped = False
    just_died = False

    if health <= 0:
        just_died = True
        health = 100.0
        fly_pos = np.array([0.0, 1.0, 0.0])
    else:
        if is_in_pain and motor_activity > 0:
            target_pos = stop_pain_btn_pos
        elif is_in_diarrhea and motor_activity > 0:
            target_pos = stop_diarrhea_btn_pos
        else:
            target_pos = None

        if target_pos is not None:
            direction = target_pos - fly_pos
            distance = np.linalg.norm(direction)

            if distance > 0:
                direction = direction / distance
                ideal_rotation = np.arctan2(direction[0], direction[2])
                fly_rotation = ideal_rotation + np.random.uniform(-1.0, 1.0)

            forward = np.array([np.sin(fly_rotation), 0, np.cos(fly_rotation)])
            fly_pos += forward * fly_speed

            if distance < 1.0:
                if is_in_pain: pain_stopped = True
                if is_in_diarrhea: diarrhea_stopped = True

        else:
            fly_rotation += np.random.uniform(-0.8, 0.8)
            forward = np.array([np.sin(fly_rotation), 0, np.cos(fly_rotation)])
            fly_pos += forward * (fly_speed * 0.4)

        if fly_pos[0] <= -38 or fly_pos[0] >= 38 or fly_pos[2] <= -38 or fly_pos[2] >= 38:
            fly_rotation += np.pi + np.random.uniform(-0.5, 0.5)

        fly_pos[0] = np.clip(fly_pos[0], -40, 40)
        fly_pos[2] = np.clip(fly_pos[2], -40, 40)

    return JSON({
        "x": float(fly_pos[0]),
        "y": float(fly_pos[1]),
        "z": float(fly_pos[2]),
        "rotation": float(fly_rotation),
        "health": float(health),
        "pain_stopped": pain_stopped,
        "diarrhea_stopped": diarrhea_stopped,
        "just_died": just_died
    })

output.register_callback('compute_tick', compute_tick)


html_code = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
    <style>
        body { margin: 0; overflow: hidden; font-family: sans-serif; background: #000; }
        #canvas-container { width: 100%; height: 500px; position: relative; }
        #ui-container { position: absolute; top: 10px; left: 10px; z-index: 100; pointer-events: none; display: flex; flex-direction: column; gap: 10px; width: 300px;}
        .action-btn { color: white; border: 2px solid white; padding: 15px 30px; font-size: 16px; border-radius: 5px; cursor: pointer; user-select: none; font-weight: bold; pointer-events: auto; transition: 0.2s;}
        .action-btn:active { transform: scale(0.95); }
        #zap-btn { background: #44aa44; }
        #poop-btn { background: #44aa44; }
        #health-container { width: 100%; height: 20px; background: #333; border: 2px solid white; border-radius: 5px; overflow: hidden; position: relative;}
        #health-bar { width: 100%; height: 100%; background: #00ff00; transition: 0.1s; }
        #health-text { position: absolute; top: 0; left: 50%; transform: translateX(-50%); font-weight: bold; color: white; font-size: 14px; text-shadow: 1px 1px 2px #000; }
        #status { color: white; margin-top: 10px; font-weight: bold; font-size: 18px; text-shadow: 2px 2px 4px #000000; }
        #error-log { color: #ffaa00; margin-top: 5px; font-family: monospace; font-size: 12px; }
    </style>
</head>
<body>
    <div id="canvas-container">
        <div id="ui-container">
            <div id="health-container">
                <div id="health-bar"></div>
                <div id="health-text">HP: 100</div>
            </div>
            <button id="zap-btn" class="action-btn">APPLY PAIN (OFF)</button>
            <button id="poop-btn" class="action-btn">APPLY DIARRHEA (OFF)</button>
            <div id="status">Status: Connecting to Hemibrain Connectome...</div>
            <div id="error-log"></div>
        </div>
    </div>
    <script>
        let isPainActive = false;
        let isDiarrheaActive = false;
        const zapBtn = document.getElementById('zap-btn');
        const poopBtn = document.getElementById('poop-btn');
        const statusText = document.getElementById('status');
        const healthBar = document.getElementById('health-bar');
        const healthText = document.getElementById('health-text');

        function updateBtnUI() {
            if (isPainActive) {
                zapBtn.innerText = "PAIN ACTIVE (ON)"; zapBtn.style.background = "#ff4444";
            } else {
                zapBtn.innerText = "APPLY PAIN (OFF)"; zapBtn.style.background = "#44aa44";
            }
            if (isDiarrheaActive) {
                poopBtn.innerText = "DIARRHEA ACTIVE (ON)"; poopBtn.style.background = "#8B4513";
            } else {
                poopBtn.innerText = "APPLY DIARRHEA (OFF)"; poopBtn.style.background = "#44aa44";
            }

            if (isPainActive && isDiarrheaActive) {
                statusText.innerText = 'Status: MULTIPLE CRISES! Prioritizing pain...'; statusText.style.color = '#ff4444';
            } else if (isPainActive) {
                statusText.innerText = 'Status: IN PAIN! Seeking green target...'; statusText.style.color = '#ff4444';
            } else if (isDiarrheaActive) {
                statusText.innerText = 'Status: DIARRHEA! Seeking orange bathroom...'; statusText.style.color = '#ffaa00';
            } else {
                statusText.innerText = 'Status: Free Wandering'; statusText.style.color = 'white';
            }
        }

        zapBtn.addEventListener('click', () => { isPainActive = !isPainActive; updateBtnUI(); });
        poopBtn.addEventListener('click', () => { isDiarrheaActive = !isDiarrheaActive; updateBtnUI(); });

        const container = document.getElementById('canvas-container');
        const scene = new THREE.Scene();
        scene.background = new THREE.Color(0x87CEEB);
        const camera = new THREE.PerspectiveCamera(60, window.innerWidth / 500, 0.1, 1000);
        camera.position.set(0, 40, 50);
        const renderer = new THREE.WebGLRenderer({ antialias: true });
        renderer.setSize(container.clientWidth, 500);
        container.appendChild(renderer.domElement);
        const controls = new THREE.OrbitControls(camera, renderer.domElement);
        const light = new THREE.DirectionalLight(0xffffff, 1.2);
        light.position.set(20, 50, 20);
        scene.add(light);
        scene.add(new THREE.AmbientLight(0x606060));

        const floorGeo = new THREE.PlaneGeometry(100, 100);
        const floorMat = new THREE.MeshStandardMaterial({ color: 0x555555 });
        const floor = new THREE.Mesh(floorGeo, floorMat);
        floor.rotation.x = -Math.PI / 2;
        scene.add(floor);
        scene.add(new THREE.GridHelper(100, 100));

        const flyGroup = new THREE.Group();
        const bodyGeo = new THREE.SphereGeometry(0.5, 16, 16);
        const bodyMat = new THREE.MeshStandardMaterial({ color: 0x111111 });
        const body = new THREE.Mesh(bodyGeo, bodyMat);
        body.scale.set(0.8, 0.8, 1.8);
        flyGroup.add(body);

        const eyeGeo = new THREE.SphereGeometry(0.2, 16, 16);
        const eyeMat = new THREE.MeshStandardMaterial({ color: 0xff0000 });
        const eyeL = new THREE.Mesh(eyeGeo, eyeMat); eyeL.position.set(0.3, 0.3, 0.6);
        const eyeR = new THREE.Mesh(eyeGeo, eyeMat); eyeR.position.set(-0.3, 0.3, 0.6);
        flyGroup.add(eyeL); flyGroup.add(eyeR);

        const wingGeo = new THREE.PlaneGeometry(2.0, 0.8);
        const wingMat = new THREE.MeshStandardMaterial({ color: 0xeeeeee, transparent: true, opacity: 0.6, side: THREE.DoubleSide });
        const wingL = new THREE.Mesh(wingGeo, wingMat); wingL.position.set(1.0, 0.4, 0); wingL.rotation.x = Math.PI / 2;
        const wingR = new THREE.Mesh(wingGeo, wingMat); wingR.position.set(-1.0, 0.4, 0); wingR.rotation.x = Math.PI / 2;
        flyGroup.add(wingL); flyGroup.add(wingR);
        scene.add(flyGroup);

        const btnGeo = new THREE.CylinderGeometry(0.8, 0.8, 0.5, 32);
        const stopPainBtn = new THREE.Mesh(btnGeo, new THREE.MeshStandardMaterial({ color: 0x00ff00 }));
        stopPainBtn.position.set(25, 0.25, 25);
        scene.add(stopPainBtn);
        const stopDiarrheaBtn = new THREE.Mesh(btnGeo, new THREE.MeshStandardMaterial({ color: 0xff8c00 }));
        stopDiarrheaBtn.position.set(-25, 0.25, -25);
        scene.add(stopDiarrheaBtn);

        const poopArray = [];
        const corpses = [];
        const dyingAnimQueue = [];
        const poopGeo = new THREE.SphereGeometry(0.3, 8, 8);
        const poopMaterial = new THREE.MeshStandardMaterial({ color: 0x5c4033 });

        function createMinecraftCorpse(sourceGroup) {
            const corpse = sourceGroup.clone();
            corpse.traverse((child) => {
                if (child.isMesh) {
                    child.material = child.material.clone();
                    child.material.color.setHex(0xff0000);
                }
            });
            scene.add(corpse);
            dyingAnimQueue.push({ mesh: corpse, timer: 0.0, startRotZ: corpse.rotation.z });
            corpses.push(corpse);
            if (corpses.length > 30) { scene.remove(corpses[0]); corpses.shift(); }
        }

        let targetPos = new THREE.Vector3(0, 1, 0);
        let targetRot = 0;
        let wingAngle = 0;

        async function updateSimulation() {
            try {
                if (typeof google !== 'undefined' && google.colab) {
                    const result = await google.colab.kernel.invokeFunction('compute_tick', [isPainActive, isDiarrheaActive], {});
                    let state = result.data['application/json'];

                    if (state) {
                        if (state.just_died) {
                            createMinecraftCorpse(flyGroup);
                            isPainActive = false; isDiarrheaActive = false;
                            updateBtnUI();
                            targetPos.set(0, 1, 0); flyGroup.position.set(0, 1, 0);
                        } else {
                            targetPos.set(state.x, state.y, state.z);
                        }

                        targetRot = state.rotation;
                        healthBar.style.width = Math.max(0, state.health) + '%';
                        healthText.innerText = "HP: " + Math.round(state.health);
                        if (state.health > 50) healthBar.style.background = "#00ff00";
                        else if (state.health > 25) healthBar.style.background = "#ffff00";
                        else healthBar.style.background = "#ff0000";

                        let changed = false;
                        if (state.pain_stopped && isPainActive) { isPainActive = false; changed = true; }
                        if (state.diarrhea_stopped && isDiarrheaActive) { isDiarrheaActive = false; changed = true; }
                        if (changed) updateBtnUI();

                        if (isPainActive) bodyMat.color.setHex(0x770000);
                        else if (isDiarrheaActive) bodyMat.color.setHex(0x4a3424);
                        else bodyMat.color.setHex(0x111111);

                        if (isDiarrheaActive) {
                            const newPoop = new THREE.Mesh(poopGeo, poopMaterial);
                            newPoop.position.set(flyGroup.position.x, 0.15, flyGroup.position.z);
                            scene.add(newPoop); poopArray.push(newPoop);
                            if (poopArray.length > 200) { scene.remove(poopArray[0]); poopArray.shift(); }
                        }
                    }
                }
            } catch (err) {
                document.getElementById('error-log').innerText = "Processing connectome math...";
            }
            setTimeout(updateSimulation, 50);
        }

        updateBtnUI();
        updateSimulation();

        function animate() {
            requestAnimationFrame(animate);
            flyGroup.position.lerp(targetPos, 0.2);
            let rotDiff = targetRot - flyGroup.rotation.y;
            while (rotDiff > Math.PI) rotDiff -= Math.PI * 2;
            while (rotDiff < -Math.PI) rotDiff += Math.PI * 2;
            flyGroup.rotation.y += rotDiff * 0.15;
            wingAngle += 1.2;
            wingL.rotation.y = Math.sin(wingAngle) * 0.6;
            wingR.rotation.y = -Math.sin(wingAngle) * 0.6;

            for (let i = dyingAnimQueue.length - 1; i >= 0; i--) {
                let anim = dyingAnimQueue[i];
                anim.timer += 0.1;
                anim.mesh.rotation.z = anim.startRotZ + Math.min(anim.timer, 1.0) * (Math.PI / 2);
                anim.mesh.position.y = Math.max(0.3, 1.0 - (anim.timer * 0.5));
                if (anim.timer >= 1.0) {
                    anim.mesh.traverse((child) => { if (child.isMesh && child.material.color) child.material.color.setHex(0x440000); });
                    dyingAnimQueue.splice(i, 1);
                }
            }
            controls.update(); renderer.render(scene, camera);
        }
        animate();
    </script>
</body>
</html>
"""


display(HTML(html_code))